# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Lecture 12: checking how well a model works

This notebook accompanies Sections 9.1--9.3. A low training loss is
not itself a guarantee about population risk. We reserve an independent
final test split, handle undefined metric denominators explicitly, and
fit every preprocessing operation inside each cross-validation fold.


In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

np.set_printoptions(precision=4, suppress=True)

def hoeffding_interval(mean_loss, sample_size, B=1, alpha=0.05):
    radius = B * np.sqrt(np.log(2 / alpha) / (2 * sample_size))
    return max(0, mean_loss - radius), min(B, mean_loss + radius), radius


## What a held-out test set tells us

After training is fixed, $\widehat g$ is fixed. If $m$ independent
test losses lie in $[0,B]$, then
$$
\mathbb P\left(
|\widehat R_m^{\rm test}(\widehat g)-R(\widehat g)|>\epsilon
\mid T_n\right)
\leq2\exp(-2m\epsilon^2/B^2).
$$
The confidence half-width is
$B\sqrt{\log(2/\alpha)/(2m)}$. This does not imply that training
risk is always smaller than test or population risk.


In [ ]:
digits = load_digits()
X, y = digits.data, (digits.target >= 5).astype(int)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=2026
)
print("development/test sizes:", len(X_dev), len(X_test))


## Cross-validation without leakage

Cross-validation uses only development data. The scaler is inside the
pipeline, so each fold estimates its mean and scale without its
held-out fold. Fitting it once before cross-validation would leak
validation information.

Fold losses are dependent because fitted models use overlapping
training observations. The i.i.d. held-out Hoeffding proof therefore
does not apply to the cross-validation mean. We make no VC-dimension
claim from these empirical scores.


In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
summary = []
for C in [0.03, 0.1, 0.3, 1.0]:
    procedure = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C, max_iter=2_000, random_state=2026),
    )
    result = cross_validate(
        procedure, X_dev, y_dev, cv=folds, scoring="accuracy",
        return_train_score=True,
    )
    summary.append((C, result["test_score"].mean(),
                    result["test_score"].std(ddof=1)))
for C, mean_score, sd_score in summary:
    print(f"C={C:>4}: mean fold accuracy={mean_score:.3f}, "
          f"fold SD={sd_score:.3f}")

selected_C = max(summary, key=lambda row: row[1])[0]
final_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=selected_C, max_iter=2_000, random_state=2026),
)
final_model.fit(X_dev, y_dev)
test_prediction = final_model.predict(X_test)
test_error = np.mean(test_prediction != y_test)
interval = hoeffding_interval(test_error, len(y_test), B=1, alpha=0.05)
print("selected C:", selected_C)
print("one-time final test error:", test_error)
print("95% Hoeffding interval for population 0-1 risk:", interval[:2])


The final procedure was selected and refitted without the test data.
The test set was then used once, so the bound applies to this fixed
workflow.

## Accuracy, precision, and recall

For class $1$,
$$
\text{precision}=\mathbb P(Y=1\mid g(X)=1),\qquad
\text{recall}=\mathbb P(g(X)=1\mid Y=1).
$$
Population precision needs $\mathbb P(g(X)=1)>0$, while recall needs
$\mathbb P(Y=1)>0$. Their empirical effective sample sizes are the
random numbers predicted positive and actually positive.


In [ ]:
def binary_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, int), np.asarray(y_pred, int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    n_pred, n_pos = tp + fp, tp + fn
    return {
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "accuracy": (tp + tn) / len(y_true),
        "precision": tp / n_pred if n_pred else np.nan,
        "recall": tp / n_pos if n_pos else np.nan,
        "n_pred": n_pred, "n_pos": n_pos,
    }

metrics = binary_metrics(y_test, test_prediction)
metrics


### Confidence intervals for precision and recall

Conditional on a positive retained count, the filtered observations
are Bernoulli. Allocating $\alpha/2$ to each metric gives radius
$\sqrt{\log(4/\alpha)/(2N)}$. If an observed retained count is zero,
the sample supplies no information and we report $[0,1]$, not a
division by zero.


In [ ]:
def filtered_interval(estimate, count, alpha=0.05):
    if count == 0:
        return 0.0, 1.0
    radius = np.sqrt(np.log(4 / alpha) / (2 * count))
    return max(0, estimate - radius), min(1, estimate + radius)

precision_interval = filtered_interval(
    metrics["precision"], metrics["n_pred"]
)
recall_interval = filtered_interval(metrics["recall"], metrics["n_pos"])
print("precision/count/interval:",
      metrics["precision"], metrics["n_pred"], precision_interval)
print("recall/count/interval:",
      metrics["recall"], metrics["n_pos"], recall_interval)


## What if a denominator is zero?

A classifier that never predicts $1$ has no empirical precision
denominator. If a test set has no label $1$, empirical recall has no
denominator. Returning NaN makes an undefined ratio visible; replacing
it silently by zero changes the definition.


In [ ]:
edge_cases = {
    "never predicts 1": (
        np.array([0, 0, 1, 1]), np.array([0, 0, 0, 0])
    ),
    "always predicts 1": (
        np.array([0, 0, 1, 1]), np.array([1, 1, 1, 1])
    ),
    "no positive labels": (
        np.array([0, 0, 0, 0]), np.array([0, 1, 0, 1])
    ),
}
for name, (truth, prediction) in edge_cases.items():
    result = binary_metrics(truth, prediction)
    print(f"{name:20s}: precision={result['precision']}, "
          f"recall={result['recall']}, accuracy={result['accuracy']}")


## Check an analysis for leakage

Choose another regularization grid and repeat the development-set
cross-validation. Identify every operation that must be fitted inside
each fold; distinguish fold-to-fold standard deviation from a
confidence interval; state exactly when the final-test Hoeffding
interval is justified; and construct examples where precision or
recall is undefined. Keep the final test data untouched until the
procedure has been selected and refitted.
